# Waste Management End-to-End Notebook
This notebook runs the full pipeline:
1) Download dataset `annotations.json`
2) Download dataset images
3) Visualize samples
4) Preprocess (remap original classes to 8 target labels + split)
5) Build YOLO segmentation dataset
6) Train `yolo26n-seg`
7) Evaluate and save metrics
8) Save/export model artifacts

In [ ]:
from pathlib import Path
import json
import shutil
import os
import random
from collections import defaultdict, Counter
from io import BytesIO

import requests
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

IS_KAGGLE = Path('/kaggle').exists()
WORKING_DIR = Path('/kaggle/working') if IS_KAGGLE else Path.cwd()
INPUT_DIR = Path('/kaggle/input') if IS_KAGGLE else None

PROJECT_ROOT = WORKING_DIR

DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
YOLO_SEG_DATASET_DIR = PROCESSED_DIR / 'yolo_seg_dataset'
RUNS_DIR = PROJECT_ROOT / 'runs' / 'yolo-seg'
EXPORT_DIR = PROJECT_ROOT / 'artifacts' / 'exports' / 'yolo-seg'

for p in [DATA_DIR, RAW_DIR, PROCESSED_DIR, YOLO_SEG_DATASET_DIR, RUNS_DIR, EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

ANNOTATIONS_PATH = DATA_DIR / 'annotations.json'
DATASET_JSON_URL = 'https://huggingface.co/datasets/karimaouaouda/taco/resolve/main/annotations.json'
YOLO_SEG_MODEL = 'yolo26n-seg.pt'

def _find_kaggle_annotations_json():
    if not INPUT_DIR or not INPUT_DIR.exists():
        return None
    candidates = list(INPUT_DIR.glob('**/annotations.json'))
    return candidates[0] if candidates else None

def download_dataset(dataset_path, data_dir, show_progress=True):
    dataset_path = Path(dataset_path)
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    with dataset_path.open('r', encoding='utf-8') as f:
        annotations = json.load(f)

    images = annotations.get('images', [])
    total = len(images)
    for i, image in enumerate(images, start=1):
        file_name = image['file_name']
        url_original = image.get('flickr_url')
        if not url_original:
            continue

        file_path = data_dir / file_name
        file_path.parent.mkdir(parents=True, exist_ok=True)

        if not file_path.exists():
            response = requests.get(url_original, timeout=30)
            response.raise_for_status()
            img = Image.open(BytesIO(response.content))
            if img.getexif():
                img.save(file_path, exif=img.info.get('exif'))
            else:
                img.save(file_path)

        if show_progress and i % 200 == 0:
            print(f'Downloaded/checked {i}/{total} images')

    if show_progress:
        print(f'Finished image download/check: {total} images referenced')
    return total

def load_image(image_info, data_dir):
    return Image.open(Path(data_dir) / image_info['file_name'])

def print_dataset_summary(images, annotations, categories):
    annotated_ids = {ann['image_id'] for ann in annotations}
    unannotated = sum(1 for img in images if img['id'] not in annotated_ids)
    avg_anns = len(annotations) / len(images) if images else 0

    print('=' * 40)
    print('TACO Dataset Summary')
    print('=' * 40)
    print(f'  Total images      : {len(images)}')
    print(f'  Total annotations : {len(annotations)}')
    print(f'  Total categories  : {len(categories)}')
    print(f'  Avg anns / image  : {avg_anns:.2f}')
    print(f'  Images w/o anns   : {unannotated}')
    print('=' * 40)

def visualize_image_with_annotations(image, annotations, categories=None):
    cat_map = {c['id']: c['name'] for c in categories} if categories else {}
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    ax = plt.gca()

    for annotation in annotations:
        bbox = annotation['bbox']
        cat_id = annotation['category_id']
        label = cat_map.get(cat_id, str(cat_id))
        rect = patches.Rectangle((bbox[0], bbox[1]), bbox[2], bbox[3], linewidth=2, edgecolor='r', facecolor='none')
        ax.add_patch(rect)
        plt.text(bbox[0], bbox[1] - 10, label, color='red', fontsize=10,
                 bbox=dict(facecolor='white', alpha=0.5, edgecolor='none', pad=1))

    plt.axis('off')
    plt.show()

def visulize_simples(images, annotations, data_dir, N_SAMPLES=5, with_annotations=True, categories=None):
    sample_images = random.sample(images, min(N_SAMPLES, len(images)))
    for image_info in sample_images:
        img = load_image(image_info, data_dir)
        image_annotations = [ann for ann in annotations if ann['image_id'] == image_info['id']]
        if with_annotations:
            visualize_image_with_annotations(img, image_annotations, categories)
        else:
            plt.figure(figsize=(10, 10))
            plt.imshow(img)
            plt.title(image_info['file_name'])
            plt.axis('off')
            plt.show()

def analyze_class_distribution(categories, annotations, plot=False):
    class_counts = Counter(ann['category_id'] for ann in annotations)
    distribution = {cat['id']: class_counts[cat['id']] for cat in categories}

    if plot:
        names = [cat['name'] for cat in categories]
        counts = [distribution[cat['id']] for cat in categories]
        pairs = [(n, c) for n, c in zip(names, counts) if c > 0]
        if pairs:
            names, counts = zip(*sorted(pairs, key=lambda x: -x[1]))
        fig, ax = plt.subplots(figsize=(16, 6))
        ax.bar(range(len(names)), counts)
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels(names, rotation=90, fontsize=8)
        ax.set_ylabel('Count')
        ax.set_title('Class Distribution in TACO Dataset')
        plt.tight_layout()
        plt.show()

    return distribution

def analyze_bbox_distribution(annotations, plot=False):
    widths = [ann['bbox'][2] for ann in annotations]
    heights = [ann['bbox'][3] for ann in annotations]
    areas = [w * h for w, h in zip(widths, heights)]

    def _stats(values):
        return {'min': min(values), 'max': max(values), 'mean': sum(values) / len(values)}

    stats = {'width': _stats(widths), 'height': _stats(heights), 'area': _stats(areas)}
    print('Bounding Box Statistics:')
    for key, s in stats.items():
        print(f"  {key:7s}: min={s['min']:.1f}  max={s['max']:.1f}  mean={s['mean']:.1f}")

    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        for ax, values, label in zip(axes, [widths, heights, areas], ['BBox Width', 'BBox Height', 'BBox Area']):
            ax.hist(values, bins=50, edgecolor='black')
            ax.set_title(label)
            ax.set_xlabel('Pixels' if 'Area' not in label else 'Pixels²')
            ax.set_ylabel('Count')
        plt.suptitle('Bounding Box Size Distribution')
        plt.tight_layout()
        plt.show()

    return stats

def visualize_segmentation_masks(image, annotations, categories=None, alpha=0.45):
    import numpy as np

    cat_map = {c['id']: c['name'] for c in categories} if categories else {}
    cmap = plt.get_cmap('tab20')
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(image)

    legend_handles = {}
    for idx, ann in enumerate(annotations):
        cat_id = ann['category_id']
        label = cat_map.get(cat_id, str(cat_id))
        colour = cmap(idx % 20)

        for seg in ann.get('segmentation', []):
            if len(seg) < 6:
                continue
            poly_pts = np.array(seg).reshape(-1, 2)
            poly = patches.Polygon(poly_pts, closed=True,
                                   facecolor=(*colour[:3], alpha),
                                   edgecolor=(*colour[:3], 1.0),
                                   linewidth=1.5)
            ax.add_patch(poly)
            cx, cy = poly_pts[:, 0].mean(), poly_pts[:, 1].mean()
            ax.text(cx, cy, label, fontsize=8, ha='center', va='center',
                    color='white',
                    bbox=dict(facecolor=colour[:3], alpha=0.6, edgecolor='none', pad=1))

        if label not in legend_handles:
            legend_handles[label] = patches.Patch(facecolor=colour[:3], label=label)

    if legend_handles:
        ax.legend(handles=list(legend_handles.values()), loc='upper right', fontsize=7, framealpha=0.7)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

def visualize_segmentation_samples(images, annotations, data_dir, N_SAMPLES=5, categories=None):
    ann_by_image = defaultdict(list)
    for ann in annotations:
        ann_by_image[ann['image_id']].append(ann)

    eligible = [img for img in images if any(ann.get('segmentation') for ann in ann_by_image.get(img['id'], []))]
    if not eligible:
        print('No images with segmentation data found.')
        return

    sample = random.sample(eligible, min(N_SAMPLES, len(eligible)))
    for image_info in sample:
        img = load_image(image_info, data_dir)
        anns = ann_by_image.get(image_info['id'], [])
        print(f"Image: {image_info['file_name']} ({len(anns)} annotations)")
        visualize_segmentation_masks(img, anns, categories=categories)

def _validate_split_ratios(train_ratio, val_ratio, test_ratio):
    total = train_ratio + val_ratio + test_ratio
    if not 0 < train_ratio < 1 or not 0 < val_ratio < 1 or not 0 < test_ratio < 1:
        raise ValueError('Each split ratio must be in (0, 1)')
    if abs(total - 1.0) > 1e-8:
        raise ValueError(f'Split ratios must sum to 1.0, got {total:.4f}')

def _build_split_lists(images, train_ratio, val_ratio, seed):
    shuffled = list(images)
    random.Random(seed).shuffle(shuffled)
    total = len(shuffled)
    train_count = int(total * train_ratio)
    val_count = int(total * val_ratio)
    return {
        'train': shuffled[:train_count],
        'val': shuffled[train_count:train_count + val_count],
        'test': shuffled[train_count + val_count:],
    }

def _copy_images_for_split(images, data_dir, split_dir):
    copied = 0
    missing = 0
    for image_meta in images:
        rel = Path(image_meta['file_name'])
        src = Path(data_dir) / rel
        dst = Path(split_dir) / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if src.exists():
            shutil.copy2(src, dst)
            copied += 1
        else:
            missing += 1
    return copied, missing

def split_coco_dataset(annotations_path, data_dir, output_dir, train_ratio=0.7, val_ratio=0.1, test_ratio=0.2, seed=42, copy_images=True):
    _validate_split_ratios(train_ratio, val_ratio, test_ratio)
    annotations_path = Path(annotations_path)
    data_dir = Path(data_dir)
    output_dir = Path(output_dir)

    with annotations_path.open('r', encoding='utf-8') as f:
        dataset = json.load(f)

    images = dataset.get('images', [])
    annotations = dataset.get('annotations', [])
    categories = dataset.get('categories', [])
    scene_annotations = dataset.get('scene_annotations', [])
    licenses = dataset.get('licenses', [])
    scene_categories = dataset.get('scene_categories', [])

    split_images = _build_split_lists(images, train_ratio, val_ratio, seed)
    split_dirs = {
        'train': output_dir / 'images' / 'train',
        'val': output_dir / 'images' / 'val',
        'test': output_dir / 'images' / 'test',
    }
    for split_dir in split_dirs.values():
        split_dir.mkdir(parents=True, exist_ok=True)

    stats = {}
    for split_name, split_image_list in split_images.items():
        image_ids = {img['id'] for img in split_image_list}
        split_annotations = [ann for ann in annotations if ann.get('image_id') in image_ids]
        split_scene_annotations = [ann for ann in scene_annotations if ann.get('image_id') in image_ids]

        copied = 0
        missing = 0
        if copy_images:
            copied, missing = _copy_images_for_split(split_image_list, data_dir, split_dirs[split_name])

        payload = {
            'images': split_image_list,
            'annotations': split_annotations,
            'categories': categories,
            'scene_annotations': split_scene_annotations,
            'licenses': licenses,
            'scene_categories': scene_categories,
        }
        with (output_dir / f'annotations_{split_name}.json').open('w', encoding='utf-8') as f:
            json.dump(payload, f)

        stats[split_name] = {
            'images': len(split_image_list),
            'annotations': len(split_annotations),
            'copied_images': copied,
            'missing_images': missing,
        }

    with (output_dir / 'split_summary.json').open('w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2)
    return stats

def _count_annotations_per_category(annotations):
    counts = defaultdict(int)
    for ann in annotations:
        category_id = ann.get('category_id')
        if category_id is not None:
            counts[category_id] += 1
    return dict(counts)

def build_dataset_by_merging_rare_classes(annotations_path, data_dir, output_dir, min_instances_per_class=20, copy_images=True):
    if min_instances_per_class < 1:
        raise ValueError('min_instances_per_class must be >= 1')

    annotations_path = Path(annotations_path)
    data_dir = Path(data_dir)
    output_dir = Path(output_dir)
    processed_dir = output_dir / f'merged_min_{min_instances_per_class}'
    processed_dir.mkdir(parents=True, exist_ok=True)

    with annotations_path.open('r', encoding='utf-8') as f:
        dataset = json.load(f)

    images = dataset.get('images', [])
    annotations = dataset.get('annotations', [])
    categories = dataset.get('categories', [])
    scene_annotations = dataset.get('scene_annotations', [])
    licenses = dataset.get('licenses', [])
    scene_categories = dataset.get('scene_categories', [])

    counts_before = _count_annotations_per_category(annotations)
    if not counts_before:
        raise ValueError('No annotations found; cannot merge classes')

    all_category_ids = {cat.get('id') for cat in categories}
    rare_category_ids = {cid for cid in all_category_ids if counts_before.get(cid, 0) < min_instances_per_class}

    categories_by_id = {cat['id']: cat for cat in categories if 'id' in cat}
    super_to_category_ids = defaultdict(list)
    for cat in categories:
        if 'id' in cat:
            super_to_category_ids[cat.get('supercategory', '')].append(cat['id'])

    global_largest_category_id = max(counts_before, key=lambda cid: counts_before.get(cid, 0))

    merge_map = {}
    for rare_id in sorted(rare_category_ids):
        supercategory = categories_by_id.get(rare_id, {}).get('supercategory', '')
        peers = [cid for cid in super_to_category_ids.get(supercategory, []) if cid != rare_id]
        if peers:
            target_id = max(peers, key=lambda cid: counts_before.get(cid, 0))
        else:
            target_id = global_largest_category_id
        merge_map[rare_id] = target_id

    merged_annotations = []
    for ann in annotations:
        updated = dict(ann)
        cid = updated.get('category_id')
        if cid in merge_map:
            updated['category_id'] = merge_map[cid]
        merged_annotations.append(updated)

    used_category_ids = {ann.get('category_id') for ann in merged_annotations}
    used_image_ids = {ann.get('image_id') for ann in merged_annotations}

    merged_categories = [cat for cat in categories if cat.get('id') in used_category_ids]
    merged_images = [img for img in images if img.get('id') in used_image_ids]
    merged_scene_annotations = [ann for ann in scene_annotations if ann.get('image_id') in used_image_ids]

    copied = 0
    missing = 0
    if copy_images:
        copied, missing = _copy_images_for_split(merged_images, data_dir, processed_dir / 'images')

    merged_dataset = {
        'images': merged_images,
        'annotations': merged_annotations,
        'categories': merged_categories,
        'scene_annotations': merged_scene_annotations,
        'licenses': licenses,
        'scene_categories': scene_categories,
    }
    annotations_out = processed_dir / 'annotations_merged.json'
    with annotations_out.open('w', encoding='utf-8') as f:
        json.dump(merged_dataset, f)

    counts_after = _count_annotations_per_category(merged_annotations)
    summary = {
        'operation': 'merge_rare_classes',
        'min_instances_per_class': min_instances_per_class,
        'input_images': len(images),
        'input_annotations': len(annotations),
        'output_images': len(merged_images),
        'output_annotations': len(merged_annotations),
        'input_categories': len(categories),
        'output_categories': len(merged_categories),
        'merged_categories': len(merge_map),
        'copied_images': copied,
        'missing_images': missing,
        'output_annotations_path': str(annotations_out),
    }

    (processed_dir / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    (processed_dir / 'merge_map.json').write_text(json.dumps({'merge_map': merge_map}, indent=2), encoding='utf-8')
    (processed_dir / 'category_counts_before.json').write_text(json.dumps(counts_before, indent=2), encoding='utf-8')
    (processed_dir / 'category_counts_after.json').write_text(json.dumps(counts_after, indent=2), encoding='utf-8')

    return summary

def export_yolo_model(weights_path, export_format='onnx', output_dir=None, imgsz=640, device=None, simplify=False):
    from ultralytics import YOLO

    weights_path = Path(weights_path).resolve()
    if not weights_path.exists():
        raise FileNotFoundError(f'weights not found: {weights_path}')

    model = YOLO(str(weights_path))
    kwargs = {'format': export_format, 'imgsz': imgsz, 'simplify': simplify}
    if device is not None:
        kwargs['device'] = device

    if output_dir is not None:
        output_dir = Path(output_dir).resolve()
        output_dir.mkdir(parents=True, exist_ok=True)
        kwargs['project'] = str(output_dir)
        kwargs['name'] = ''

    exported = model.export(**kwargs)
    exported_path = Path(exported) if not isinstance(exported, Path) else exported
    if not exported_path.is_absolute():
        exported_path = Path.cwd() / exported_path
    return exported_path.resolve()

print('Environment:', 'Kaggle' if IS_KAGGLE else 'Local')
print('Working dir:', PROJECT_ROOT)
print('Annotations path:', ANNOTATIONS_PATH)

In [ ]:
# Install dependencies when running in a fresh environment
%pip install -r requirements.txt

In [ ]:
# 1) Download annotations.json (or reuse one from /kaggle/input)
from urllib.request import urlretrieve

if not ANNOTATIONS_PATH.exists():
    kaggle_annotations = _find_kaggle_annotations_json()
    if kaggle_annotations is not None:
        print(f'Using annotations.json from Kaggle input: {kaggle_annotations}')
        shutil.copy2(kaggle_annotations, ANNOTATIONS_PATH)
    else:
        print('Downloading annotations.json from Hugging Face...')
        urlretrieve(DATASET_JSON_URL, ANNOTATIONS_PATH)
else:
    print('annotations.json already exists, skipping download.')

print('annotations.json ready at:', ANNOTATIONS_PATH)

In [ ]:
# 2) Download dataset images referenced in annotations.json

num_images = download_dataset(dataset_path=str(ANNOTATIONS_PATH), data_dir=str(RAW_DIR), show_progress=False)
print(f'Images referenced in annotations: {num_images}')

In [ ]:
# Load JSON
with ANNOTATIONS_PATH.open('r', encoding='utf-8') as f:
    coco = json.load(f)

images = coco['images']
annotations = coco['annotations']
categories = coco['categories']

print('images:', len(images))
print('annotations:', len(annotations))
print('categories:', len(categories))

In [ ]:
# 3) Visualize
%matplotlib inline

print_dataset_summary(images, annotations, categories)
_ = analyze_class_distribution(categories, annotations, plot=True)
_ = analyze_bbox_distribution(annotations, plot=True)
visulize_simples(images, annotations, data_dir=str(RAW_DIR), N_SAMPLES=2, with_annotations=True, categories=categories)
visualize_segmentation_samples(images, annotations, data_dir=str(RAW_DIR), N_SAMPLES=2, categories=categories)

In [ ]:
# 4) Preprocess (remap original classes to 8 target labels + split)

TARGET_LABELS = [
    'organic_waste',
    'plastic_bottle',
    'plastic_bag',
    'rigid_plastic',
    'metal_can',
    'glass',
    'paper_cardboard',
    'mixed_waste',
]

OLD_TO_NEW_LABEL = {
    'Food waste': 'organic_waste',
    'Other plastic bottle': 'plastic_bottle',
    'Clear plastic bottle': 'plastic_bottle',
    'Plastic film': 'plastic_bag',
    'Six pack rings': 'plastic_bag',
    'Garbage bag': 'plastic_bag',
    'Other plastic wrapper': 'plastic_bag',
    'Single-use carrier bag': 'plastic_bag',
    'Polypropylene bag': 'plastic_bag',
    'Crisp packet': 'plastic_bag',
    'Plastic bottle cap': 'rigid_plastic',
    'Plastic lid': 'rigid_plastic',
    'Other plastic': 'rigid_plastic',
    'Disposable plastic cup': 'rigid_plastic',
    'Foam cup': 'rigid_plastic',
    'Other plastic cup': 'rigid_plastic',
    'Spread tub': 'rigid_plastic',
    'Tupperware': 'rigid_plastic',
    'Disposable food container': 'rigid_plastic',
    'Foam food container': 'rigid_plastic',
    'Other plastic container': 'rigid_plastic',
    'Plastic glooves': 'rigid_plastic',
    'Plastic utensils': 'rigid_plastic',
    'Squeezable tube': 'rigid_plastic',
    'Plastic straw': 'rigid_plastic',
    'Styrofoam piece': 'mixed_waste',
    'Aluminium foil': 'metal_can',
    'Aluminium blister pack': 'metal_can',
    'Metal bottle cap': 'metal_can',
    'Food Can': 'metal_can',
    'Aerosol': 'metal_can',
    'Drink can': 'metal_can',
    'Metal lid': 'metal_can',
    'Pop tab': 'metal_can',
    'Scrap metal': 'metal_can',
    'Glass bottle': 'glass',
    'Broken glass': 'glass',
    'Glass cup': 'glass',
    'Glass jar': 'glass',
    'Toilet tube': 'paper_cardboard',
    'Other carton': 'paper_cardboard',
    'Egg carton': 'paper_cardboard',
    'Drink carton': 'paper_cardboard',
    'Corrugated carton': 'paper_cardboard',
    'Meal carton': 'paper_cardboard',
    'Pizza box': 'paper_cardboard',
    'Paper cup': 'paper_cardboard',
    'Magazine paper': 'paper_cardboard',
    'Tissues': 'paper_cardboard',
    'Wrapping paper': 'paper_cardboard',
    'Normal paper': 'paper_cardboard',
    'Paper bag': 'paper_cardboard',
    'Plastified paper bag': 'paper_cardboard',
    'Paper straw': 'paper_cardboard',
    'Battery': 'mixed_waste',
    'Carded blister pack': 'mixed_waste',
    'Rope & strings': 'mixed_waste',
    'Shoe': 'mixed_waste',
    'Unlabeled litter': 'mixed_waste',
    'Cigarette': 'mixed_waste',
}


def _resolve_target_label(category):
    name = str(category.get('name', '')).strip()
    supercategory = str(category.get('supercategory', '')).strip()
    lower_name = name.lower()
    lower_supercategory = supercategory.lower()

    if name in OLD_TO_NEW_LABEL:
        return OLD_TO_NEW_LABEL[name]

    if lower_supercategory == 'bottle':
        return 'glass' if 'glass' in lower_name else 'plastic_bottle'
    if lower_supercategory == 'bottle cap':
        return 'metal_can' if 'metal' in lower_name else 'rigid_plastic'
    if lower_supercategory in {'paper', 'carton', 'paper bag'}:
        return 'paper_cardboard'
    if lower_supercategory == 'plastic bag & wrapper':
        return 'plastic_bag'
    if lower_supercategory == 'plastic container':
        return 'rigid_plastic'
    if lower_supercategory == 'cup':
        if lower_name.startswith('glass'):
            return 'glass'
        if lower_name.startswith('paper'):
            return 'paper_cardboard'
        return 'rigid_plastic'
    if lower_supercategory == 'lid':
        return 'metal_can' if 'metal' in lower_name else 'rigid_plastic'
    if lower_supercategory == 'other plastic':
        return 'rigid_plastic'
    if lower_supercategory == 'straw':
        return 'paper_cardboard' if lower_name.startswith('paper') else 'rigid_plastic'
    if lower_supercategory == 'food waste':
        return 'organic_waste'
    if lower_supercategory in {'battery', 'unlabeled litter', 'rope & strings', 'shoe', 'cigarette'}:
        return 'mixed_waste'
    if 'glass' in lower_name:
        return 'glass'
    if 'can' in lower_name or 'foil' in lower_name or 'metal' in lower_name:
        return 'metal_can'
    if 'paper' in lower_name or 'carton' in lower_name or 'tissue' in lower_name or 'box' in lower_name:
        return 'paper_cardboard'
    if 'plastic' in lower_name or 'foam' in lower_name or 'tupperware' in lower_name or 'tube' in lower_name or 'glove' in lower_name or 'utensil' in lower_name or 'straw' in lower_name:
        return 'rigid_plastic'
    return 'mixed_waste'


def remap_coco_categories(annotations_path, output_dir, target_labels, old_to_new_label):
    annotations_path = Path(annotations_path)
    output_dir = Path(output_dir)
    remap_dir = output_dir / 'remapped_8classes'
    remap_dir.mkdir(parents=True, exist_ok=True)

    with annotations_path.open('r', encoding='utf-8') as f:
        dataset = json.load(f)

    images = dataset.get('images', [])
    annotations = dataset.get('annotations', [])
    categories = dataset.get('categories', [])
    scene_annotations = dataset.get('scene_annotations', [])
    licenses = dataset.get('licenses', [])
    scene_categories = dataset.get('scene_categories', [])

    label_to_id = {label: idx for idx, label in enumerate(target_labels)}
    remapped_annotations = []
    unknown_source_categories = set()

    for annotation in annotations:
        src_category = next((category for category in categories if category.get('id') == annotation.get('category_id')), None)
        if src_category is None:
            continue

        src_name = str(src_category.get('name', '')).strip()
        if src_name in old_to_new_label:
            dst_label = old_to_new_label[src_name]
        else:
            dst_label = _resolve_target_label(src_category)
            unknown_source_categories.add(src_name)

        mapped = dict(annotation)
        mapped['category_id'] = label_to_id[dst_label]
        remapped_annotations.append(mapped)

    remapped_images = list(images)
    remapped_scene_annotations = list(scene_annotations)
    remapped_categories = [
        {
            'id': idx,
            'name': label,
            'supercategory': label,
        }
        for idx, label in enumerate(target_labels)
    ]

    copied_images = 0
    missing_images = 0
    source_images_dir = annotations_path.parent / 'raw'
    output_images_dir = remap_dir / 'images'
    output_images_dir.mkdir(parents=True, exist_ok=True)
    for image_meta in remapped_images:
        relative_path = Path(image_meta['file_name'])
        source_path = source_images_dir / relative_path
        target_path = output_images_dir / relative_path
        target_path.parent.mkdir(parents=True, exist_ok=True)
        if source_path.exists():
            shutil.copy2(source_path, target_path)
            copied_images += 1
        else:
            missing_images += 1

    remapped_dataset = {
        'images': remapped_images,
        'annotations': remapped_annotations,
        'categories': remapped_categories,
        'scene_annotations': remapped_scene_annotations,
        'licenses': licenses,
        'scene_categories': scene_categories,
    }

    remapped_annotations_path = remap_dir / 'annotations_remapped.json'
    remapped_annotations_path.write_text(json.dumps(remapped_dataset), encoding='utf-8')

    label_counts = Counter()
    for annotation in remapped_annotations:
        label_counts[annotation['category_id']] += 1

    summary = {
        'target_labels': target_labels,
        'input_images': len(images),
        'input_annotations': len(annotations),
        'output_images': len(remapped_images),
        'output_annotations': len(remapped_annotations),
        'output_categories': len(remapped_categories),
        'copied_images': copied_images,
        'missing_images': missing_images,
        'unknown_source_categories': sorted(unknown_source_categories),
        'counts_by_target_label': {
            label: int(label_counts[idx]) for idx, label in enumerate(target_labels)
        },
        'remapped_annotations_path': str(remapped_annotations_path.resolve()),
    }

    summary_path = remap_dir / 'remap_summary.json'
    summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

    return remapped_annotations_path, summary_path, summary


remapped_annotations_path, remap_summary_path, remap_summary = remap_coco_categories(
    annotations_path=ANNOTATIONS_PATH,
    output_dir=PROCESSED_DIR,
    target_labels=TARGET_LABELS,
    old_to_new_label=OLD_TO_NEW_LABEL,
)

print('remapped_annotations_path:', remapped_annotations_path)
print('remap_summary_path:', remap_summary_path)
print('remap_summary:', json.dumps(remap_summary, indent=2))

TRAIN_RATIO = 0.7
VAL_RATIO = 0.1
TEST_RATIO = 0.2
SEED = 42

split_stats = split_coco_dataset(
    annotations_path=remapped_annotations_path,
    data_dir=PROCESSED_DIR / 'remapped_8classes' / 'images',
    output_dir=PROCESSED_DIR,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED,
    copy_images=True,
)

print('split_stats:', split_stats)

In [ ]:
# 5) Build YOLO segmentation dataset from processed COCO splits
from collections import defaultdict

def _ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

def _normalize_point(x, y, w, h):
    x = max(0.0, min(1.0, x / float(w)))
    y = max(0.0, min(1.0, y / float(h)))
    return x, y

def build_yolo_seg_dataset(processed_dir: Path, output_dir: Path):
    split_names = ['train', 'val', 'test']

    train_payload = json.loads((processed_dir / 'annotations_train.json').read_text(encoding='utf-8'))
    categories = sorted(train_payload.get('categories', []), key=lambda c: int(c['id']))
    category_id_to_yolo_id = {int(c['id']): i for i, c in enumerate(categories)}
    class_names = [str(c.get('name', f'class_{i}')) for i, c in enumerate(categories)]

    images_root = output_dir / 'images'
    labels_root = output_dir / 'labels'
    _ensure_dir(images_root)
    _ensure_dir(labels_root)

    summary = {}

    for split in split_names:
        payload = json.loads((processed_dir / f'annotations_{split}.json').read_text(encoding='utf-8'))
        images = payload.get('images', [])
        anns = payload.get('annotations', [])

        anns_by_image = defaultdict(list)
        for ann in anns:
            anns_by_image[int(ann['image_id'])].append(ann)

        src_split_dir = processed_dir / 'images' / split
        dst_images_dir = images_root / split
        dst_labels_dir = labels_root / split
        _ensure_dir(dst_images_dir)
        _ensure_dir(dst_labels_dir)

        copied_images = 0
        missing_images = 0
        written_label_files = 0
        written_instances = 0

        for img in images:
            img_id = int(img['id'])
            rel = Path(img['file_name'])
            src_img = src_split_dir / rel
            dst_img = dst_images_dir / rel
            dst_lbl = (dst_labels_dir / rel).with_suffix('.txt')
            _ensure_dir(dst_img.parent)
            _ensure_dir(dst_lbl.parent)

            if not src_img.exists():
                missing_images += 1
                continue

            shutil.copy2(src_img, dst_img)
            copied_images += 1

            img_w = int(img['width'])
            img_h = int(img['height'])
            label_lines = []

            for ann in anns_by_image.get(img_id, []):
                cat_id = int(ann['category_id'])
                if cat_id not in category_id_to_yolo_id:
                    continue

                segs = ann.get('segmentation', [])
                if not isinstance(segs, list):
                    continue

                cls = category_id_to_yolo_id[cat_id]
                for seg in segs:
                    if not isinstance(seg, list) or len(seg) < 6:
                        continue

                    points = []
                    for i in range(0, len(seg), 2):
                        if i + 1 >= len(seg):
                            break
                        x, y = _normalize_point(float(seg[i]), float(seg[i + 1]), img_w, img_h)
                        points.extend([f'{x:.6f}', f'{y:.6f}'])

                    if len(points) >= 6:
                        label_lines.append(f"{cls} " + ' '.join(points))

            dst_lbl.write_text('\n'.join(label_lines), encoding='utf-8')
            written_label_files += 1
            written_instances += len(label_lines)

        summary[split] = {
            'images': len(images),
            'copied_images': copied_images,
            'missing_images': missing_images,
            'label_files': written_label_files,
            'instances': written_instances,
        }

    dataset_yaml = output_dir / 'dataset.yaml'
    yaml_lines = [
        f'path: {str(output_dir.resolve())}',
        'train: images/train',
        'val: images/val',
        'test: images/test',
        f'nc: {len(class_names)}',
        'names:',
    ]
    yaml_lines.extend([f'  {i}: {name}' for i, name in enumerate(class_names)])
    dataset_yaml.write_text('\n'.join(yaml_lines) + '\n', encoding='utf-8')

    summary_path = output_dir / 'conversion_summary.json'
    summary_payload = {
        'dataset_yaml': str(dataset_yaml.resolve()),
        'classes': class_names,
        'splits': summary,
    }
    summary_path.write_text(json.dumps(summary_payload, indent=2), encoding='utf-8')

    return dataset_yaml, summary_path, summary_payload

dataset_yaml_path, conversion_summary_path, conversion_summary = build_yolo_seg_dataset(
    processed_dir=PROCESSED_DIR,
    output_dir=YOLO_SEG_DATASET_DIR
)

print('dataset_yaml:', dataset_yaml_path)
print('conversion_summary:', conversion_summary_path)
print(json.dumps(conversion_summary, indent=2)[:1200])

In [ ]:
# 6) Tune + Train YOLO26n-seg with Optuna
from ultralytics import YOLO
import mlflow
from pathlib import Path
import os
import json
import optuna

def _normalize_mlflow_uri(var_name: str):
    value = os.getenv(var_name)
    if not value:
        return
    value = value.strip()
    if not value:
        return
    if "://" in value:
        return
    os.environ[var_name] = Path(value).expanduser().resolve().as_uri()

os.environ.setdefault("MLFLOW_TRACKING_URI", str((PROJECT_ROOT / "mlruns").resolve().as_uri()))
os.environ.setdefault("MLFLOW_REGISTRY_URI", os.environ["MLFLOW_TRACKING_URI"])

_normalize_mlflow_uri("MLFLOW_TRACKING_URI")
_normalize_mlflow_uri("MLFLOW_REGISTRY_URI")

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_registry_uri(os.environ["MLFLOW_REGISTRY_URI"])

# Final training config
EPOCHS = 50
IMGSZ = 640
BATCH = 16
EXPERIMENT_NAME = 'waste-seg-yolo26n'

# Optuna search config (keep this modest for local runs)
OPTUNA_TRIALS = 10
TUNE_EPOCHS = 12
TUNE_EXPERIMENT_NAME = f'{EXPERIMENT_NAME}-optuna'

def objective(trial: optuna.Trial) -> float:
    params = {
        'lr0': trial.suggest_float('lr0', 1e-4, 5e-2, log=True),
        'lrf': trial.suggest_float('lrf', 0.01, 1.0),
        'momentum': trial.suggest_float('momentum', 0.7, 0.98),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
        'warmup_epochs': trial.suggest_float('warmup_epochs', 0.0, 5.0),
        'box': trial.suggest_float('box', 3.0, 10.0),
        'cls': trial.suggest_float('cls', 0.1, 3.0),
        'dfl': trial.suggest_float('dfl', 0.5, 3.0),
    }

    tune_model = YOLO(YOLO_SEG_MODEL)
    tune_name = f"{TUNE_EXPERIMENT_NAME}-trial{trial.number}"
    tune_model.train(
        data=str(dataset_yaml_path),
        epochs=TUNE_EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        project=str(RUNS_DIR),
        name=tune_name,
        verbose=False,
        **params,
    )

    tune_weights = RUNS_DIR / tune_name / 'weights' / 'best.pt'
    if not tune_weights.exists():
        raise RuntimeError(f'Tuning weights not found for trial {trial.number}: {tune_weights}')

    tune_eval = YOLO(str(tune_weights))
    tune_metrics = tune_eval.val(data=str(dataset_yaml_path), split='val', verbose=False)
    score = float(tune_metrics.results_dict.get('metrics/seg/mAP50-95(B)', 0.0))

    trial.set_user_attr('weights', str(tune_weights.resolve()))
    trial.set_user_attr('score_name', 'metrics/seg/mAP50-95(B)')
    return score

study = optuna.create_study(direction='maximize', study_name=f'{EXPERIMENT_NAME}-study')
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best_params = study.best_trial.params
print('Best trial:', study.best_trial.number)
print('Best score:', study.best_value)
print('Best params:', json.dumps(best_params, indent=2))

best_params_path = RUNS_DIR / f'{EXPERIMENT_NAME}_optuna_best_params.json'
best_params_path.write_text(json.dumps(best_params, indent=2), encoding='utf-8')
print('Saved best params:', best_params_path)

# Final train with best hyperparameters
model = YOLO(YOLO_SEG_MODEL)
train_results = model.train(
    data=str(dataset_yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(RUNS_DIR),
    name=EXPERIMENT_NAME,
    **best_params,
)

run_dir = RUNS_DIR / EXPERIMENT_NAME
weights_dir = run_dir / 'weights'
best_weights = weights_dir / 'best.pt'
last_weights = weights_dir / 'last.pt'
trained_weights = best_weights if best_weights.exists() else last_weights
print('Run dir:', run_dir)
print('Trained weights:', trained_weights)

In [ ]:
# 7) Evaluate and save metrics
import pandas as pd

eval_model = YOLO(str(trained_weights))
val_metrics = eval_model.val(data=str(dataset_yaml_path), split='val')
test_metrics = eval_model.val(data=str(dataset_yaml_path), split='test')

def metrics_to_dict(metrics_obj):
    if hasattr(metrics_obj, 'results_dict') and isinstance(metrics_obj.results_dict, dict):
        return dict(metrics_obj.results_dict)
    if isinstance(metrics_obj, dict):
        return dict(metrics_obj)
    return {'value': str(metrics_obj)}

val_dict = metrics_to_dict(val_metrics)
test_dict = metrics_to_dict(test_metrics)

optuna_params = {}
if 'best_params_path' in globals() and Path(best_params_path).exists():
    optuna_params = json.loads(Path(best_params_path).read_text(encoding='utf-8'))

metrics_payload = {
    'model': YOLO_SEG_MODEL,
    'trained_weights': str(trained_weights.resolve()),
    'dataset_yaml': str(dataset_yaml_path.resolve()),
    'optuna_best_params': optuna_params,
    'val_metrics': val_dict,
    'test_metrics': test_dict,
}

metrics_json_path = run_dir / 'metrics_summary.json'
metrics_json_path.write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')

rows = []
for split_name, metric_dict in [('val', val_dict), ('test', test_dict)]:
    for k, v in metric_dict.items():
        rows.append({'split': split_name, 'metric': k, 'value': v})

metrics_csv_path = run_dir / 'metrics_summary.csv'
pd.DataFrame(rows).to_csv(metrics_csv_path, index=False)

print('Metrics JSON:', metrics_json_path)
print('Metrics CSV :', metrics_csv_path)
pd.DataFrame(rows).head(20)

In [ ]:
# 8) Save model artifacts (weights copy + ONNX export)

saved_weights_path = EXPORT_DIR / f'{EXPERIMENT_NAME}_best.pt'
saved_weights_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(trained_weights, saved_weights_path)

exported_model_path = export_yolo_model(
    weights_path=trained_weights,
    export_format='onnx',
    output_dir=EXPORT_DIR,
    imgsz=IMGSZ,
    simplify=True,
)

artifacts_summary = {
    'run_dir': str(run_dir.resolve()),
    'saved_weights': str(saved_weights_path.resolve()),
    'exported_model': str(Path(exported_model_path).resolve()),
    'metrics_json': str((run_dir / 'metrics_summary.json').resolve()),
    'metrics_csv': str((run_dir / 'metrics_summary.csv').resolve()),
}

artifacts_summary_path = run_dir / 'artifacts_summary.json'
artifacts_summary_path.write_text(json.dumps(artifacts_summary, indent=2), encoding='utf-8')

print(json.dumps(artifacts_summary, indent=2))